In [0]:
from pyspark.sql.functions import sum,count, avg , max , col , current_timestamp , cast  , count 
from delta import DeltaTable
from pyspark.sql. types import  DecimalType

In [0]:
#  reading the  table as a data frame 
destinationDF = spark.read.table("nyctaxi.inbound.aggregatedAvginformation")

In [0]:
inboundDF = spark.read.table('nyctaxi.inbound.nyc_taxi_inbound')
inboundTableObject = DeltaTable.forName(spark ,'nyctaxi.inbound.nyc_taxi_inbound')
destinationTableObject =   DeltaTable.forName(spark ,'nyctaxi.inbound.aggregatedAvginformation')

In [0]:
inboundDF  = spark.read.table('nyctaxi.inbound.nyc_taxi_inbound')
if destinationDF.count() == 0:
    fullLoadAggregated = inboundDF.groupBy("hvfhs_license_num").agg(
                                count(col('hvfhs_license_num')).alias('total_trips') ,        
                                sum(col("base_passenger_fare")).cast(DecimalType(34,2)).alias('totalBaseFee') , 
                                avg(col("trip_miles")).cast(DecimalType(34,2)).alias('avgTripMiles') , 
                                max(col("pickup_datetime")).alias('lastTripTime')
                                                )

    destinationTableObject.alias("target").merge(
    fullLoadAggregated.alias('source') , 
     col("target.hvfhs_license_num") == col("source.hvfhs_license_num")
).whenNotMatchedInsert(
    values = {
            'hvfhs_license_num' : col('source.hvfhs_license_num'),
            'total_trips':col('source.total_trips'),
            'total_fare_amount':col('source.totalBaseFee'),
            'avg_trip_distance':col('source.avgTripMiles'),
            'last_trip_date':col('source.lastTripTime') , 
            'inserted_time': current_timestamp()
    }
).execute()

    spark.sql("""
                INSERT INTO  nyctaxi.inbound.watermark
                        (
                        table_name ,
                        last_updatedValue
                        )

                        SELECT 'nyc_taxi_inbound', max(request_datetime) FROM nyctaxi.inbound.nyc_taxi_inbound
                                
            """)

else: 
    newDataReceived  = spark.sql("""
            SELECT * FROM nyctaxi.inbound.nyc_taxi_inbound
            where request_datetime > (
            SELECT last_updatedValue FROM  nyctaxi.inbound.watermark
     )
 """
)
    tempAggregatedResult = (
            newDataReceived.groupBy("hvfhs_license_num")
            .agg(
                count(col('hvfhs_license_num')).cast(DecimalType(34,2)).alias('totalTrips') ,
                sum(col("base_passenger_fare")).cast(DecimalType(34,2)).alias('totalBaseFee'),
                avg(col("trip_miles")).cast(DecimalType(34,2)).alias('avgTripMiles'),
                max(col("pickup_datetime")).alias('lastTripTime')
            )
        )
    aggregatedavginformationTObject  = DeltaTable.forName(spark,'nyctaxi.inbound.aggregatedavginformation')
    aggregatedavginformationTObject.alias('destination').merge(
    tempAggregatedResult.alias('source') , 
    col("destination.hvfhs_license_num") == col("source.hvfhs_license_num")
).whenMatchedUpdate(
 set = {
       'total_trips' :col('source.totalTrips') + col('destination.total_trips') , 
       'total_fare_amount':col('source.totalBaseFee') + col('destination.total_fare_amount'),
       'avg_trip_distance':col('source.avgTripMiles') + col('destination.avg_trip_distance'),
       'last_trip_date':col('source.lastTripTime') ,
       'updated_at': current_timestamp()
 }
).whenNotMatchedInsert(
    values = {

                     'hvfhs_license_num' : col('source.hvfhs_license_num'),
                     'total_trips':col('source.totalTrips'),   
                     'total_fare_amount':col('source.totalBaseFee'),
                     'avg_trip_distance':col('source.avgTripMiles'),
                     'last_trip_date':col('source.lastTripTime') , 
                     'inserted_time': current_timestamp()

    }  
).execute()
    spark.sql("""
                  update nyctaxi.inbound.watermark
                  set last_updatedValue = (
                  SELECT max(request_datetime) FROM nyctaxi.inbound.nyc_taxi_inbound
                  ) , 
                
                  updated_timestamo = current_timestamp()
              """)

    # updating the information to the 